In [ ]:
from __future__ import annotations

import argparse
import atexit
import copy
import csv
import json
import logging
import random
import re
import shutil
import sys
import time
import types
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

sys.path.insert(0, str(Path("").resolve()))

from scripts.convert_gsm8k_to_sft import parse_gsm8k_answer
from scripts.eval_sft_inference import evaluate_gsm8k
from src.rl.prm_scorer import ProcessRewardScorer
from src.sft.solution_format import extract_final_answer_numeric_str
from src.utils.attn_backend import select_attn_implementation
from src.rl.math_environment_curriculum import CurriculumMathEnvironment
from src.config.prompts import create_generator_messages

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s - %(message)s",
)
logger = logging.getLogger(__name__)

In [ ]:
class TeeStream:
    def __init__(self, primary, secondary):
        self.primary = primary
        self.secondary = secondary

    def write(self, data: str) -> int:
        self.primary.write(data)
        self.secondary.write(data)
        return len(data)

    def flush(self) -> None:
        self.primary.flush()
        self.secondary.flush()

    def isatty(self) -> bool:
        return getattr(self.primary, "isatty", lambda: False)()

    def fileno(self) -> int:
        return self.primary.fileno()


def _add_file_logging(log_path: Path) -> logging.FileHandler:
    fh = logging.FileHandler(log_path, mode="a", encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(logging.Formatter(
        "%(asctime)s %(levelname)-8s %(name)s - %(message)s"
    ))
    logging.getLogger().addHandler(fh)
    return fh


if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

In [ ]:
def _infer_eval_dataset_name(data_path: str) -> str:
    stem = Path(data_path).stem.lower()
    if "aqua" in stem:
        return "AQuA-RAT"
    if "math" in stem:
        return "MATH"
    if "gsm" in stem:
        return "GSM8K"
    return Path(data_path).stem


def load_gsm8k(path: str) -> List[Dict[str, str]]:
    pairs: List[Dict[str, str]] = []
    p = Path(path)
    if not p.exists():
        logger.warning("Training data not found at %s", path)
        return pairs
    with p.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            question = ""
            gold = ""
            if "question" in rec and "answer" in rec:
                question = rec["question"].strip()
                _, gold = parse_gsm8k_answer(str(rec["answer"]))
            elif "messages" in rec:
                user_text = ""
                asst_text = ""
                for msg in rec["messages"]:
                    if msg.get("role") == "user" and not user_text:
                        user_text = msg.get("content", "").strip()
                    elif msg.get("role") == "assistant" and not asst_text:
                        asst_text = msg.get("content", "")
                if "Problem:" in user_text:
                    question = user_text.split("Problem:", 1)[1].strip()
                else:
                    question = user_text
                answer_str = extract_final_answer_numeric_str(asst_text) or ""
                gold = answer_str.strip()
            if question and gold:
                pairs.append({"question": question, "gold_final": gold})
    logger.info("Loaded %d QA pairs from %s", len(pairs), path)
    return pairs


def _extract_boxed(text: str) -> Optional[str]:
    m = re.search(r"\\boxed\{([^}]*)\}", text)
    return m.group(1).strip() if m else None


def _boxed_to_numeric(answer: str) -> Optional[str]:
    ans = answer.strip()
    try:
        return str(int(ans))
    except ValueError:
        pass
    try:
        v = float(ans)
        return str(int(v)) if v == int(v) else f"{v:.4f}"
    except ValueError:
        pass
    m = re.fullmatch(r"\\frac\{(\d+)\}\{(\d+)\}", ans)
    if m:
        num, den = int(m.group(1)), int(m.group(2))
        if den:
            v = num / den
            return str(int(v)) if v == int(v) else f"{v:.4f}"
    m = re.fullmatch(r"(\d+)/(\d+)", ans)
    if m:
        num, den = int(m.group(1)), int(m.group(2))
        if den:
            v = num / den
            return str(int(v)) if v == int(v) else f"{v:.4f}"
    return None


def load_math_dataset(
    local_path: Optional[str] = None,
    cache_path: str = "data/math/math_numeric.jsonl",
    max_difficulty: int = 3,
) -> List[Dict[str, str]]:
    for candidate in filter(None, [local_path, cache_path]):
        p = Path(candidate)
        if p.exists():
            pairs: List[Dict[str, str]] = []
            with p.open(encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        try:
                            pairs.append(json.loads(line))
                        except json.JSONDecodeError:
                            pass
            if pairs:
                logger.info("Loaded %d MATH pairs from %s", len(pairs), p)
                return pairs

    logger.info(
        "MATH dataset not found locally — downloading from HuggingFace "
        "(qwedsacf/competition_math, difficulty \u2264 %d, numeric answers only)...",
        max_difficulty,
    )
    _HF_SOURCES = [
        ("qwedsacf/competition_math", {}),
        ("lighteval/MATH-Hard",       {"name": "default"}),
    ]
    ds = None
    for hf_name, hf_kwargs in _HF_SOURCES:
        try:
            from datasets import load_dataset
            ds = load_dataset(hf_name, split="train", trust_remote_code=True, **hf_kwargs)
            logger.info("Loaded HuggingFace dataset: %s (%d items)", hf_name, len(ds))
            break
        except Exception as exc:
            logger.warning("Could not load %s: %s \u2014 trying next source.", hf_name, exc)
    if ds is None:
        logger.warning(
            "All MATH dataset sources failed. Proceeding with GSM8K only. "
            "To load offline: download from https://github.com/hendrycks/math "
            "and pass --math-data <path_to_jsonl>."
        )
        return []

    pairs = []
    for item in ds:
        level_str = item.get("level", "Level 5")
        try:
            level = int(level_str.split()[-1])
        except (ValueError, IndexError):
            level = 5
        if level > max_difficulty:
            continue
        question = item.get("problem", "").strip()
        solution = item.get("solution", "")
        boxed    = _extract_boxed(solution)
        if not boxed:
            continue
        numeric  = _boxed_to_numeric(boxed)
        if not numeric:
            continue
        pairs.append({"question": question, "gold_final": numeric})

    if pairs:
        out_p = Path(cache_path)
        out_p.parent.mkdir(parents=True, exist_ok=True)
        with out_p.open("w", encoding="utf-8") as f:
            for p_item in pairs:
                f.write(json.dumps(p_item) + "\n")
        logger.info("Cached %d MATH numeric pairs to %s", len(pairs), out_p)
    else:
        logger.warning("No MATH pairs passed the numeric filter \u2014 check the dataset.")

    return pairs

In [ ]:
import re as _re

_FINAL_ANSWER_RE = _re.compile(r"final answer[:\s]*([^\n]+)", _re.I)

_PAL_TOPICS     = frozenset({"arithmetic", "algebra", "prealgebra", "grounded"})
_SYMPY_TOPICS   = frozenset({
    "number_theory", "intermediate_algebra", "precalculus",
    "counting_and_probability",
})
_EXCLUDE_TOPICS = frozenset({"geometry"})


def _extract_final_answer(solution: str) -> Optional[str]:
    m = _FINAL_ANSWER_RE.search(solution)
    return m.group(1).strip() if m else None


def _pal_eval(answer_str: str) -> Optional[float]:
    try:
        val = eval(answer_str, {"__builtins__": {}}, {})  # noqa: S307
        f = float(val)
        return None if f != f else f
    except Exception:
        return None


def _sympy_eval(answer_str: str) -> Optional[float]:
    try:
        from sympy import sympify, N as _N
        f = float(_N(sympify(answer_str), 15))
        return None if f != f else f
    except Exception:
        return None


def _verify_self_play_answer(
    solutions: List[str],
    target_topic: str,
    target_difficulty: float,
) -> bool:
    topic = target_topic.lower().replace(" ", "_")
    if topic in _EXCLUDE_TOPICS or target_difficulty >= 4.0:
        return False
    answers: List[float] = []
    for sol in solutions:
        raw = _extract_final_answer(sol)
        if raw is None:
            continue
        val: Optional[float]
        if topic in _PAL_TOPICS or target_difficulty <= 2:
            val = _pal_eval(raw) or _sympy_eval(raw)
        elif topic in _SYMPY_TOPICS:
            val = _sympy_eval(raw) or _pal_eval(raw)
        else:
            val = _pal_eval(raw) or _sympy_eval(raw)
        if val is not None:
            answers.append(round(val, 6))
    if not answers:
        return False
    majority = max(set(answers), key=answers.count)
    return answers.count(majority) >= max(1, len(solutions) // 2)


def compute_grounded_reward(
    question: str,
    solution: str,
    gold_final: str,
    math_env: CurriculumMathEnvironment,
) -> Dict[str, float]:
    result = math_env.compute_grounded_reward(
        question=question,
        solution=solution,
        gold_final=gold_final,
    )
    return {
        "combined_score":  float(result.get("combined_score",  0.0)),
        "step_accuracy":   float(result.get("step_accuracy",   0.0)),
        "lccp":            float(result.get("lccp",            0.0)),
        "prm_mean_score":  float(result.get("prm_mean_score",  0.0)),
        "prm_final_score": float(result.get("prm_final_score", 0.0)),
        "gt_match":        bool(result.get("gt_match",         False)),
        "format_score":    float(result.get("format_score",    0.0)),
    }


def compute_self_play_reward(
    question: str,
    solution: str,
    target_topic: str,
    target_difficulty: float,
    math_env: CurriculumMathEnvironment,
) -> Tuple[float, float, float, Dict]:
    result = math_env.compute_reward(
        question=question,
        solution=solution,
        target_topic=target_topic,
        target_difficulty=target_difficulty,
    )
    combined  = float(result["combined_score"])
    sol_score = result.get("solution_metrics", {})
    s_reward  = float(sol_score.get("overall_score", 0.0)) if isinstance(sol_score, dict) else 0.0
    q_metrics_raw = result.get("question_metrics", {}) or {}
    q_reward = float(result.get("effective_question_reward", q_metrics_raw.get("overall_score", 0.0)))
    q_metrics: Dict = {
        "overall_score":  q_reward,
        "topic_match":    float(q_metrics_raw.get("topic_match",       0.0)),
        "difficulty_fit": float(q_metrics_raw.get("difficulty_score",  0.0)),
        "clarity":        float(q_metrics_raw.get("clarity",           0.0)),
        "novelty":        float(q_metrics_raw.get("novelty_combined",  0.0)),
        "solvability":    float(q_metrics_raw.get("solvability_score", 0.0)),
        "sp_chain_integrity_score": result.get("sp_chain_integrity_score"),
    }
    return combined, q_reward, s_reward, q_metrics

In [ ]:
@torch.no_grad()
def generate_question(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    instruction: str,
    max_new_tokens: int,
    device: torch.device,
    temperature: float = 0.85,
) -> str:
    messages = create_generator_messages(instruction)
    try:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        system = messages[0]["content"]
        user = messages[1]["content"]
        prompt = f"{system}\n\n{user}\n"
    enc = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    prompt_len = enc["input_ids"].shape[1]
    stop_ids: List[int] = []
    if tokenizer.eos_token_id is not None:
        stop_ids.append(tokenizer.eos_token_id)
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end, int) and im_end not in stop_ids:
        stop_ids.append(im_end)
    out = model.generate(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.95,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=stop_ids or None,
        use_cache=True,
    )
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()


def _build_stop_token_ids(tokenizer: AutoTokenizer) -> List[int]:
    stop_ids: List[int] = []
    if tokenizer.eos_token_id is not None:
        stop_ids.append(tokenizer.eos_token_id)
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end_id, int) and im_end_id not in stop_ids:
        stop_ids.append(im_end_id)
    return stop_ids or None  # type: ignore[return-value]


@torch.no_grad()
def generate_questions_batched(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    instruction: str,
    K_q: int,
    max_new_tokens: int,
    temperature: float,
    device: torch.device,
) -> Tuple[List[str], List[torch.Tensor], List[torch.Tensor], List[torch.Tensor]]:
    messages = create_generator_messages(instruction)
    try:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        prompt = f"{system}\n\n{instruction}\n"
    stop_ids = _build_stop_token_ids(tokenizer)
    pad_id: int = (
        tokenizer.pad_token_id
        if tokenizer.pad_token_id is not None
        else tokenizer.eos_token_id
    )
    enc = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    prompt_len: int = enc["input_ids"].shape[1]
    input_ids_batch = enc["input_ids"].expand(K_q, -1).contiguous()
    attn_mask_batch = enc["attention_mask"].expand(K_q, -1).contiguous()
    model.eval()
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids_batch,
            attention_mask=attn_mask_batch,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            pad_token_id=pad_id,
            eos_token_id=stop_ids,
            use_cache=True,
        )
    questions: List[str] = []
    input_ids_list: List[torch.Tensor] = []
    response_masks: List[torch.Tensor] = []
    pad_id_t = torch.tensor(pad_id, device=device, dtype=out.dtype)
    for i in range(K_q):
        full_ids = out[i]
        response_section = full_ids[prompt_len:]
        mask = torch.zeros(full_ids.shape[0], dtype=torch.bool, device=device)
        mask[prompt_len:] = response_section != pad_id_t
        question = tokenizer.decode(response_section, skip_special_tokens=True).strip()
        questions.append(question)
        input_ids_list.append(full_ids)
        response_masks.append(mask)
    old_log_probs: List[torch.Tensor] = []
    with torch.no_grad():
        attn_mask_lp = (out != pad_id_t)
        attn_mask_lp[:, :prompt_len] = True
        batch_logits = model(
            input_ids=out,
            attention_mask=attn_mask_lp.long(),
            use_cache=False,
            return_dict=True,
        ).logits
        for i in range(K_q):
            full_ids = out[i]
            mask = response_masks[i]
            shift_logits = batch_logits[i, :-1]
            shift_labels  = full_ids[1:]
            shift_mask    = mask[1:]
            lp_tokens = F.log_softmax(shift_logits, dim=-1)[
                torch.arange(shift_logits.size(0), device=device),
                shift_labels,
            ]
            resp_lps = lp_tokens[shift_mask]
            old_log_probs.append(
                resp_lps.sum().detach() if resp_lps.numel() > 0
                else torch.tensor(0.0, device=device)
            )
    return questions, input_ids_list, response_masks, old_log_probs


def generate_solutions_batched(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    K: int,
    max_new_tokens: int,
    temperature: float,
    device: torch.device,
) -> Tuple[List[str], List[torch.Tensor], List[torch.Tensor], List[torch.Tensor]]:
    stop_ids = _build_stop_token_ids(tokenizer)
    pad_id: int = (
        tokenizer.pad_token_id
        if tokenizer.pad_token_id is not None
        else tokenizer.eos_token_id
    )
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        padding=False,
        truncation=True,
        max_length=1024,
    ).to(device)
    prompt_len: int = enc["input_ids"].shape[1]
    input_ids_batch = enc["input_ids"].expand(K, -1).contiguous()
    attn_mask_batch = enc["attention_mask"].expand(K, -1).contiguous()
    model.eval()
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids_batch,
            attention_mask=attn_mask_batch,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=pad_id,
            eos_token_id=stop_ids,
            use_cache=True,
        )
    solutions: List[str] = []
    input_ids_list: List[torch.Tensor] = []
    response_masks: List[torch.Tensor] = []
    pad_id_t = torch.tensor(pad_id, device=device, dtype=out.dtype)
    for i in range(K):
        full_ids = out[i]
        response_section = full_ids[prompt_len:]
        mask = torch.zeros(full_ids.shape[0], dtype=torch.bool, device=device)
        mask[prompt_len:] = response_section != pad_id_t
        solution = tokenizer.decode(response_section, skip_special_tokens=True)
        solutions.append(solution)
        input_ids_list.append(full_ids)
        response_masks.append(mask)
    old_log_probs: List[torch.Tensor] = []
    with torch.no_grad():
        attn_mask_lp = (out != pad_id_t)
        attn_mask_lp[:, :prompt_len] = True
        batch_logits = model(
            input_ids=out,
            attention_mask=attn_mask_lp.long(),
            use_cache=False,
            return_dict=True,
        ).logits
        for i in range(K):
            full_ids = out[i]
            mask = response_masks[i]
            shift_logits = batch_logits[i, :-1]
            shift_labels  = full_ids[1:]
            shift_mask    = mask[1:]
            lp_tokens = F.log_softmax(shift_logits, dim=-1)[
                torch.arange(shift_logits.size(0), device=device),
                shift_labels,
            ]
            resp_lps = lp_tokens[shift_mask]
            old_log_probs.append(
                resp_lps.sum().detach() if resp_lps.numel() > 0
                else torch.tensor(0.0, device=device)
            )
    return solutions, input_ids_list, response_masks, old_log_probs


def compute_sequence_log_prob(
    model: AutoModelForCausalLM,
    input_ids: torch.Tensor,
    response_mask: torch.Tensor,
) -> torch.Tensor:
    ids = input_ids.unsqueeze(0)
    outputs = model(input_ids=ids, use_cache=False, return_dict=True)
    logits = outputs.logits[0]
    shift_logits = logits[:-1]
    shift_labels = input_ids[1:]
    shift_mask = response_mask[1:]
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs[
        torch.arange(shift_logits.size(0), device=shift_logits.device),
        shift_labels,
    ]
    response_log_probs = token_log_probs[shift_mask]
    if response_log_probs.numel() == 0:
        return torch.tensor(0.0, requires_grad=True, device=input_ids.device)
    return response_log_probs.sum()

In [ ]:
def grpo_loss_for_group(
    model: AutoModelForCausalLM,
    input_ids_list: List[torch.Tensor],
    response_masks: List[torch.Tensor],
    rewards: List[float],
    old_log_probs: List[torch.Tensor],
    clip_eps: float = 0.2,
    kl_coef: float = 0.0,
    ref_model: Optional[AutoModelForCausalLM] = None,
    eps: float = 1e-8,
) -> Optional[torch.Tensor]:
    rewards_arr = np.array(rewards, dtype=np.float32)
    std_r = rewards_arr.std()
    if std_r < eps:
        return None
    mean_r = rewards_arr.mean()
    advantages = (rewards_arr - mean_r) / (std_r + eps)
    advantages = np.clip(advantages, -5.0, 5.0)
    _device = next(model.parameters()).device
    group_loss = torch.tensor(0.0, device=_device)
    n_valid = 0
    model.train()
    for ids, mask, adv, old_lp in zip(
        input_ids_list, response_masks, advantages, old_log_probs
    ):
        new_lp = compute_sequence_log_prob(model, ids, mask)
        n_response = int(mask[1:].sum().item())
        if n_response == 0:
            continue
        adv_t = torch.tensor(adv, dtype=new_lp.dtype, device=_device)
        if clip_eps > 0:
            ratio = torch.exp(new_lp - old_lp.to(_device).detach())
            surr_unclipped = ratio * adv_t / n_response
            surr_clipped   = (
                torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps)
                * adv_t / n_response
            )
            loss_i = -torch.min(surr_unclipped, surr_clipped)
        else:
            loss_i = -(adv_t * new_lp / n_response)
        if kl_coef > 0.0 and ref_model is not None:
            with torch.no_grad():
                ref_lp = compute_sequence_log_prob(ref_model, ids, mask)
            kl_per_token = (new_lp - ref_lp.to(_device).detach()) / n_response
            loss_i = loss_i + kl_coef * kl_per_token
        group_loss = group_loss + loss_i
        n_valid += 1
    if n_valid == 0:
        return None
    return group_loss / n_valid

In [ ]:
def _log_eval_result(label: str, res: Dict, best: Optional[float]) -> None:
    cs      = float(res.get("combined_score",  0.0))
    cr      = float(res.get("correct_rate",    0.0))
    step_a  = float(res.get("step_accuracy",   0.0))
    lccp    = float(res.get("lccp",            0.0))
    prm     = float(res.get("prm_mean",        0.0))
    prm_fin = float(res.get("prm_final",       0.0))
    fmt     = float(res.get("format_mean",     0.0))
    n_sc    = int(res.get("n_scored", res.get("total", 0)))
    fa_acc  = float(res.get("final_answer_accuracy", cr))
    pak     = res.get("pass_at_k")
    pak_k   = int(res.get("pass_at_k_k", 4))
    best_str = f" (best={best:.4f})" if best is not None else ""
    logger.info("Training Score  [%s]: %.4f%s  |  n=%d", label, cs, best_str, n_sc)
    logger.info(
        "  Components    : 0.50\u00d7correct(%.1f%%) + 0.40\u00d7process + 0.10\u00d7fmt(%.3f)",
        100 * cr, fmt,
    )
    logger.info(
        "  Process score : prm_mean=%.3f  prm_final=%.3f  \u2192 weighted=%.3f",
        prm, prm_fin, 0.60 * prm_fin + 0.40 * prm,
    )
    logger.info("  Step accuracy : %.1f%%  (bag-of-steps: fraction of steps PRM >0.5)", 100 * step_a)
    logger.info(
        "  Chain integrity (LCCP): %.1f%%  \u2190 fraction of steps before first failure\n"
        "    [LCCP=100%% \u2192 all steps correct; LCCP=0%% \u2192 first step wrong]",
        100 * lccp,
    )
    if pak is not None:
        logger.info(
            "  pass@%d (T=0.8): %.1f%%  |  greedy correct: %.1f%%  "
            "\u2190 ceiling vs floor gap",
            pak_k, 100 * pak, 100 * cr,
        )
    logger.info("  (debug) final-answer accuracy: %.1f%%", 100 * fa_acc)


def evaluate_policy(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    eval_data_path: str,
    max_samples: int,
    max_new_tokens: int,
    math_env: Optional[Any] = None,
    pass_at_k: int = 4,
) -> Dict[str, object]:
    if not Path(eval_data_path).exists():
        return {"accuracy": 0.0, "combined_score": 0.0, "total": 0}
    model.eval()
    reward_fn = None
    if math_env is not None:
        import logging as _log_mod
        _mec_logger  = _log_mod.getLogger("src.rl.math_environment_curriculum")
        _prm_logger  = _log_mod.getLogger("src.rl.prm_scorer")

        def reward_fn(question: str, solution: str, gold: str) -> Dict:
            _old_mec = _mec_logger.level
            _old_prm = _prm_logger.level
            _mec_logger.setLevel(_log_mod.WARNING)
            _prm_logger.setLevel(_log_mod.WARNING)
            try:
                return math_env.compute_grounded_reward(question, solution, gold)
            finally:
                _mec_logger.setLevel(_old_mec)
                _prm_logger.setLevel(_old_prm)

    results = evaluate_gsm8k(
        model=model,
        tokenizer=tokenizer,
        data_path=eval_data_path,
        max_samples=max_samples,
        max_new_tokens=max_new_tokens,
        reward_fn=reward_fn,
        pass_at_k=pass_at_k,
        dataset_name=_infer_eval_dataset_name(eval_data_path),
    )
    model.train()
    return results

In [ ]:
args = argparse.Namespace(
    base_model="checkpoints/dual_task_v1",
    output_dir="checkpoints/grpo",
    gsm8k_data="data/sft/gsm8k_sft.jsonl",
    eval_data_path="data/sft/gsm8k_test.jsonl",
    num_iterations=30,
    group_size=8,
    q_group_size=2,
    questions_per_iter=16,
    learning_rate=5e-6,
    max_new_tokens=800,
    temperature=0.8,
    eval_every=5,
    eval_max_samples=100,
    eval_max_new_tokens=800,
    eval_pass_at_k=0,
    use_prm=True,
    prm_model="Qwen/Qwen2.5-Math-PRM-7B",
    skip_initial_eval=False,
    run_name=None,
    max_grad_norm=0.5,
    kl_coef=0.04,
    math_data=None,
    math_mix_ratio=0.3,
    math_mix_ratio_late=0.5,
    math_ramp_start=8,
    math_max_difficulty=3,
    clip_eps=0.2,
    warmup_iters=8,
    min_lr_ratio=0.1,
    difficulty_alpha=3.0,
    overlong_filter=True,
    save_every=5,
    keep_last=3,
    self_play_ratio=0.70,
    min_warmup=6,
    selfplay_gt_thresh=0.55,
    selfplay_grounded_thresh=0.60,
    selfplay_step_thresh=0.65,
    selfplay_ramp_iters=18,
    grounded_floor=0.50,
    extractor_model="Qwen/Qwen2.5-0.5B-Instruct",
    extraction_cache="data/extraction_cache.json",
)

In [ ]:
run_name = args.run_name or f"grpo_{datetime.now():%Y%m%d_%H%M%S}"
out_dir = Path(args.output_dir) / run_name
out_dir.mkdir(parents=True, exist_ok=True)

log_dir = Path("logs") / "grpo" / run_name
log_dir.mkdir(parents=True, exist_ok=True)

console_log_path = log_dir / "console_output.log"
_console_log_file = console_log_path.open("a", encoding="utf-8", buffering=1)

_file_handler = _add_file_logging(console_log_path)

_original_stdout = sys.stdout
_original_stderr = sys.stderr
sys.stdout = TeeStream(_original_stdout, _console_log_file)
sys.stderr = TeeStream(_original_stderr, _console_log_file)

logger.info("=" * 70)
logger.info("GRPO run: %s", run_name)
logger.info("Checkpoints : %s", out_dir)
logger.info("Logs        : %s", log_dir)
logger.info("Console log : %s", console_log_path)
logger.info("=" * 70)

(log_dir / "config.json").write_text(
    json.dumps(vars(args), indent=2, default=str), encoding="utf-8"
)

_metrics_csv_path = log_dir / "metrics.csv"
_csv_file: Optional[Any] = None
_csv_writer: Optional[Any] = None

def _append_metrics_csv(row: Dict[str, Any]) -> None:
    global _csv_file, _csv_writer
    flat = {
        k: (f"{v:.6f}" if isinstance(v, float) else v)
        for k, v in row.items()
    }
    if _csv_writer is None:
        _csv_file = _metrics_csv_path.open("w", newline="", encoding="utf-8")
        _csv_writer = csv.DictWriter(
            _csv_file,
            fieldnames=list(flat.keys()),
            extrasaction="ignore",
        )
        _csv_writer.writeheader()
    _csv_writer.writerow(flat)
    _csv_file.flush()

def _teardown_logging() -> None:
    sys.stdout = _original_stdout
    sys.stderr = _original_stderr
    logging.getLogger().removeHandler(_file_handler)
    if not getattr(_file_handler.stream, "closed", False):
        _file_handler.close()
    if _csv_file is not None and not getattr(_csv_file, "closed", False):
        _csv_file.close()
    if not _console_log_file.closed:
        _console_log_file.close()

atexit.register(_teardown_logging)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
attn_impl = select_attn_implementation()
logger.info("Device: %s | attn: %s", device, attn_impl)
if torch.cuda.is_available():
    _gpu = torch.cuda.get_device_properties(0)
    logger.info(
        "GPU: %s | %.1f GB VRAM | capability sm_%d%d",
        _gpu.name, _gpu.total_memory / 1e9, _gpu.major, _gpu.minor,
    )
logger.info(
    "Run config: K=%d K_q=%d N=%d lr=%.1e T=%.2f max_new=%d | "
    "clip_eps=%.2f kl_coef=%.4f warmup=%d | diff_alpha=%.1f | "
    "self_play=%.0f%% grounded=%.0f%% | "
    "math_mix=%.0f%% math_maxdiff=%d | overlong_filter=%s | "
    "eval_every=%d eval_N=%d | grad_clip=%.2f save_every=%d keep_last=%d | "
    "question_GRPO=%s",
    args.group_size, args.q_group_size, args.questions_per_iter, args.learning_rate,
    args.temperature, args.max_new_tokens,
    args.clip_eps, args.kl_coef, args.warmup_iters,
    args.difficulty_alpha,
    100 * args.self_play_ratio, 100 * (1 - args.self_play_ratio),
    100 * args.math_mix_ratio, args.math_max_difficulty,
    args.overlong_filter,
    args.eval_every, args.eval_max_samples,
    args.max_grad_norm, args.save_every, args.keep_last,
    f"ENABLED (K_q={args.q_group_size})" if args.q_group_size > 1 else "disabled",
)

In [ ]:
logger.info("Loading model from %s ...", args.base_model)
tokenizer = AutoTokenizer.from_pretrained(args.base_model, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if tokenizer.chat_template is None:
    _base_model_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"
    _meta_file = Path(args.base_model) / "pipeline_meta.json"
    if _meta_file.exists():
        _meta = json.loads(_meta_file.read_text(encoding="utf-8"))
        _base_model_name = _meta.get("base_model", _base_model_name)
    logger.info(
        "Tokenizer has no chat_template; loading from base model %s", _base_model_name
    )
    try:
        _base_tok = AutoTokenizer.from_pretrained(_base_model_name, trust_remote_code=True)
        if _base_tok.chat_template is not None:
            tokenizer.chat_template = _base_tok.chat_template
            logger.info("Chat template loaded successfully.")
    except Exception as _e:
        logger.warning("Could not load chat template from base model: %s", _e)

if "transformers.integrations.tensor_parallel" not in sys.modules:
    sys.modules["transformers.integrations.tensor_parallel"] = types.ModuleType(
        "tensor_parallel"
    )

model_path = Path(args.base_model)
is_adapter = (model_path / "adapter_config.json").exists()

load_kwargs = dict(
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map={"": device},
    trust_remote_code=True,
    attn_implementation=attn_impl,
)

if is_adapter:
    _meta_path = model_path / "pipeline_meta.json"
    _base_for_weights = "Qwen/Qwen2.5-Math-1.5B-Instruct"
    if _meta_path.exists():
        _base_for_weights = json.loads(
            _meta_path.read_text(encoding="utf-8")
        ).get("base_model", _base_for_weights)
    logger.info("Detected PEFT adapter \u2014 loading base %s then merging %s",
                _base_for_weights, args.base_model)
    _base = AutoModelForCausalLM.from_pretrained(_base_for_weights, **load_kwargs)
    model = PeftModel.from_pretrained(_base, args.base_model).merge_and_unload()
    model = model.to(device)
else:
    model = AutoModelForCausalLM.from_pretrained(args.base_model, **load_kwargs)

params_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
for p in model.parameters():
    p.requires_grad_(True)
params_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
if params_before == 0 and params_after > 0:
    logger.warning(
        "All parameters were frozen on load (PEFT merge_and_unload bug). "
        "Re-enabled requires_grad \u2014 any prior frozen runs were training nothing."
    )

flash_active = attn_impl == "flash_attention_2"
if not flash_active:
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    if hasattr(model, "config"):
        model.config.use_cache = False
    logger.info("Gradient checkpointing ENABLED (use_reentrant=False, use_cache=False).")
else:
    logger.info(
        "Flash-Attn 2 active \u2014 gradient checkpointing OFF "
        "(Flash already gives O(T) attention memory)."
    )

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
logger.info(
    "Trainable parameters: %s / %s (%.1f%%)",
    f"{n_params:,}", f"{n_total:,}", 100.0 * n_params / max(n_total, 1),
)

In [ ]:
ref_model: Optional[AutoModelForCausalLM] = None
if args.kl_coef > 0.0:
    logger.info(
        "Creating frozen reference policy (kl_coef=%.4f, ~%.1f GB VRAM)...",
        args.kl_coef, sum(p.numel() for p in model.parameters()) * 2 / 1e9,
    )
    ref_model = copy.deepcopy(model)
    ref_model.requires_grad_(False)
    ref_model.eval()
    logger.info("Reference policy ready.")
else:
    logger.info("KL coef = 0 \u2014 no reference policy created.")

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=args.learning_rate,
    fused=torch.cuda.is_available(),
)

from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
_n_warmup = max(1, args.warmup_iters)
_n_total  = max(1, args.num_iterations)
_n_decay  = max(1, _n_total - _n_warmup)
_min_lr   = args.learning_rate * args.min_lr_ratio
_warmup_sched = LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=_n_warmup,
)
_cosine_sched = CosineAnnealingLR(
    optimizer,
    T_max=_n_decay,
    eta_min=_min_lr,
)
scheduler = SequentialLR(
    optimizer,
    schedulers=[_warmup_sched, _cosine_sched],
    milestones=[_n_warmup],
)
logger.info(
    "LR schedule: %.1e warmup(%d iters) \u2192 cosine decay(%d iters, min=%.1e)",
    args.learning_rate, _n_warmup, _n_decay, _min_lr,
)

In [ ]:
gsm8k_pairs = load_gsm8k(args.gsm8k_data)
if not gsm8k_pairs:
    logger.error("No training data found at %s \u2014 cannot train. Exiting.", args.gsm8k_data)
    raise SystemExit(1)

math_pairs: List[Dict[str, str]] = []
if args.math_mix_ratio > 0.0:
    math_pairs = load_math_dataset(
        local_path=args.math_data,
        max_difficulty=args.math_max_difficulty,
    )
    if math_pairs:
        logger.info(
            "MATH mixing: %.0f%% MATH (%d problems) + %.0f%% GSM8K (%d problems)",
            100 * args.math_mix_ratio, len(math_pairs),
            100 * (1 - args.math_mix_ratio), len(gsm8k_pairs),
        )
    else:
        logger.warning("No MATH pairs loaded \u2014 using GSM8K only.")

qa_pairs = gsm8k_pairs

prm: Optional[ProcessRewardScorer] = None
if args.use_prm:
    try:
        prm = ProcessRewardScorer(
            model_name=args.prm_model,
            device=device,
            load_in_4bit=True,
        )
        logger.info("PRM loaded: %s (4-bit)", args.prm_model)
    except Exception as exc:
        logger.warning("PRM load failed (%s); running without PRM.", exc)

from src.rl.unified_accuracy import StepChainExtractor, UnifiedAccuracyCalculator
_extractor = StepChainExtractor(
    model_name=args.extractor_model,
    device=str(device),
    cache_path=args.extraction_cache,
)
_unified_calc = UnifiedAccuracyCalculator(extractor=_extractor, question_evaluator=None)
logger.info(
    "Unified accuracy calculator ready (extractor=%s, cache=%s)",
    args.extractor_model,
    args.extraction_cache or "none",
)
logger.info("Warming up step-chain extractor (eager load)...")
_extractor.warmup()
logger.info("Extractor warmup complete")

math_env = CurriculumMathEnvironment(
    policy_model=model,
    value_model=None,
    tokenizer=tokenizer,
    reference_questions=[p["question"] for p in gsm8k_pairs],
    grounded_qa_pairs=qa_pairs,
    prm_scorer=prm,
    max_solution_tokens=args.max_new_tokens,
    device=device,
    unified_accuracy_calc=_unified_calc,
)
_unified_calc.question_evaluator = math_env.question_evaluator

In [ ]:
from collections import defaultdict
_q_wins:     Dict[str, int] = defaultdict(int)
_q_attempts: Dict[str, int] = defaultdict(int)

def _question_key(q: str) -> str:
    import hashlib
    return hashlib.md5(q.encode(), usedforsecurity=False).hexdigest()

def _sample_by_difficulty(
    pool: List[Dict[str, str]], n: int, alpha: float
) -> List[Dict[str, str]]:
    if alpha <= 0.0:
        return random.sample(pool, min(n, len(pool)))
    weights = []
    for qa in pool:
        key = _question_key(qa["question"])
        att = _q_attempts[key]
        if att == 0:
            w = 0.75
        else:
            win_rate = _q_wins[key] / att
            info = 1.0 - abs(win_rate - 0.5) * 2.0
            w = max(info ** alpha, 0.05)
        weights.append(w)
    total_w = sum(weights)
    probs = [w / total_w for w in weights]
    chosen = np.random.choice(
        len(pool), size=min(n, len(pool)), replace=False, p=probs
    )
    return [pool[i] for i in chosen]

metrics_log: List[Dict] = []

if not args.skip_initial_eval:
    logger.info("=" * 70)
    logger.info("INITIAL EVALUATION (Iteration 0)")
    logger.info("=" * 70)
    initial_eval = evaluate_policy(
        model, tokenizer,
        args.eval_data_path, args.eval_max_samples, args.eval_max_new_tokens,
        math_env=math_env,
        pass_at_k=args.eval_pass_at_k,
    )
    _log_eval_result("INITIAL (iter 0)", initial_eval, best=None)
    metrics_log.append({"iteration": 0, **initial_eval})
    best_accuracy  = float(initial_eval.get("accuracy",     0.0))
    best_combined  = float(initial_eval.get("combined_score", 0.0))
    best_prm_mean  = float(initial_eval.get("prm_mean",     0.0))
else:
    best_accuracy = 0.0
    best_combined = 0.0
    best_prm_mean = 0.0

In [ ]:
from enum import Enum, auto as _auto

class _Phase(Enum):
    GROUNDED_ONLY = _auto()
    SELFPLAY_RAMP = _auto()
    CONTINUOUS    = _auto()

_phase: _Phase = _Phase.GROUNDED_ONLY
_selfplay_iterations: int = 0
_selfplay_suspended: bool = False
_effective_sp_ratio: float = 0.0

_use_chain_as_primary: bool = False
_chain_prm_correlation: float = 0.0
_extraction_success_rate: float = 0.0
_rolling_chain_scores:  List[float] = []
_rolling_prm_scores:    List[float] = []
_rolling_successes:     List[int]   = []
_CALIB_WINDOW = 50
_CALIB_MAX    = 200
_SHADOW_EVERY   = 4
_shadow_extract_counter: int = 0

for iteration in range(1, args.num_iterations + 1):
    iter_start = time.perf_counter()
    logger.info("=" * 70)
    logger.info("GRPO ITERATION %d/%d", iteration, args.num_iterations)
    logger.info("=" * 70)

    _effective_math_ratio = args.math_mix_ratio
    if args.math_mix_ratio_late is not None and iteration > args.math_ramp_start:
        _ramp_progress = min(1.0, (iteration - args.math_ramp_start) / 10.0)
        _effective_math_ratio = (
            args.math_mix_ratio
            + _ramp_progress * (args.math_mix_ratio_late - args.math_mix_ratio)
        )

    if math_pairs and _effective_math_ratio > 0.0:
        n_math  = max(1, round(args.questions_per_iter * _effective_math_ratio))
        n_gsm8k = max(1, args.questions_per_iter - n_math)
        math_batch  = _sample_by_difficulty(math_pairs,  n_math,  alpha=args.difficulty_alpha)
        gsm8k_batch = _sample_by_difficulty(gsm8k_pairs, n_gsm8k, alpha=args.difficulty_alpha)
        questions_batch = math_batch + gsm8k_batch
        random.shuffle(questions_batch)
    else:
        questions_batch = _sample_by_difficulty(
            gsm8k_pairs, args.questions_per_iter, alpha=args.difficulty_alpha
        )

    cur_lr = optimizer.param_groups[0]["lr"]
    _anneal_frac = min(1.0, (iteration - 1) / max(1, args.num_iterations - 1))
    _annealed_temp = args.temperature * (1.0 - 0.5 * _anneal_frac)
    logger.info(
        "LR this iteration: %.2e | T=%.3f | MATH ratio=%.0f%%",
        cur_lr, _annealed_temp, 100 * _effective_math_ratio,
    )

    all_rewards:   List[float] = []
    all_q_rewards: List[float] = []
    _grounded_rewards:   List[float] = []
    _sp_rewards:         List[float] = []
    _grounded_step_accs: List[float] = []
    _grounded_lccps:     List[float] = []
    _grounded_gt_matches: List[bool] = []
    _chain_arith_scores:     List[float] = []
    _chain_dep_scores:       List[float] = []
    _chain_integrity_scores: List[float] = []
    _sp_chain_scores:        List[float] = []
    _skipped_zero_var:   int = 0
    _qc_topic:      List[float] = []
    _qc_diff:       List[float] = []
    _qc_clarity:    List[float] = []
    _qc_novelty:    List[float] = []
    _qc_solvability: List[float] = []

    skipped = 0
    n_groups = 0
    n_self_play = 0
    q_gen_attempts  = 0
    q_gen_valid     = 0
    q_quality_good  = 0
    total_loss_val = 0.0

    if _phase == _Phase.GROUNDED_ONLY:
        _effective_sp_ratio = 0.0
    elif _phase == _Phase.SELFPLAY_RAMP:
        _grounded_anchor = max(0.30, 1.0 - (_selfplay_iterations / max(1, args.selfplay_ramp_iters)))
        _effective_sp_ratio = 1.0 - _grounded_anchor
    else:
        _effective_sp_ratio = args.self_play_ratio

    if _selfplay_suspended:
        _effective_sp_ratio = 0.0

    n_self_play_target = int(round(len(questions_batch) * _effective_sp_ratio))

    _all_indices = list(range(len(questions_batch)))
    random.shuffle(_all_indices)
    _self_play_indices = set(_all_indices[:n_self_play_target])

    optimizer.zero_grad()

    pbar = tqdm(questions_batch, desc=f"Iter {iteration} GRPO groups", unit="q")
    for _group_idx, qa in enumerate(pbar):

        use_self_play = _group_idx in _self_play_indices

        if use_self_play:
            instruction, target_topic, target_difficulty = math_env.sample_instruction()

            if target_difficulty >= 4.0:
                use_self_play = False

            q_gen_attempts += 1

            if args.q_group_size > 1:
                _q_temp = min(0.90, _annealed_temp + 0.05)
                q_cands, q_ids_all, q_masks_all, q_olps_all = generate_questions_batched(
                    model=model,
                    tokenizer=tokenizer,
                    instruction=instruction,
                    K_q=args.q_group_size,
                    max_new_tokens=128,
                    temperature=_q_temp,
                    device=device,
                )
                _valid_q = [
                    (q, ids, mask, olp)
                    for q, ids, mask, olp
                    in zip(q_cands, q_ids_all, q_masks_all, q_olps_all)
                    if len(q.strip()) >= 10
                ]
                if not _valid_q:
                    logger.debug("Two-phase SP: all %d question candidates too short, skipping.", args.q_group_size)
                    skipped += 1
                    continue
                q_gen_valid += 1
                n_self_play += 1

                _question_agg_rewards: List[float] = []
                _q_total_loss_val: float = 0.0

                for _q_text, _q_ids, _q_mask, _q_olp in _valid_q:
                    solution_prompt = math_env.format_solution_prompt(_q_text)
                    sols_q, ids_q, masks_q, olps_q = generate_solutions_batched(
                        model=model,
                        tokenizer=tokenizer,
                        prompt=solution_prompt,
                        K=args.group_size,
                        max_new_tokens=args.max_new_tokens,
                        temperature=_annealed_temp,
                        device=device,
                    )
                    if args.overlong_filter:
                        _vf = [
                            t for t in zip(sols_q, ids_q, masks_q, olps_q)
                            if int(t[2].sum().item()) < args.max_new_tokens
                        ]
                        if _vf:
                            sols_q, ids_q, masks_q, olps_q = map(list, zip(*_vf))  # type: ignore[assignment]
                        else:
                            skipped += 1
                            _question_agg_rewards.append(0.0)
                            continue

                    _sol_rewards: List[float] = []
                    for _sol in sols_q:
                        _r, _q_rew, _, _q_met = compute_self_play_reward(
                            question=_q_text,
                            solution=_sol,
                            target_topic=target_topic,
                            target_difficulty=target_difficulty,
                            math_env=math_env,
                        )
                        _sol_rewards.append(_r)
                        all_q_rewards.append(_q_rew)
                        _qc_topic.append(_q_met["topic_match"])
                        _qc_diff.append(_q_met["difficulty_fit"])
                        _qc_clarity.append(_q_met["clarity"])
                        _qc_novelty.append(_q_met["novelty"])
                        _qc_solvability.append(_q_met["solvability"])

                    all_rewards.extend(_sol_rewards)
                    _sp_rewards.extend(_sol_rewards)

                    _q_agg = float(np.mean(_sol_rewards))
                    _question_agg_rewards.append(_q_agg)

                    _sol_loss = grpo_loss_for_group(
                        model=model,
                        input_ids_list=ids_q,
                        response_masks=masks_q,
                        rewards=_sol_rewards,
                        old_log_probs=olps_q,
                        clip_eps=args.clip_eps,
                        kl_coef=args.kl_coef,
                        ref_model=ref_model,
                    )
                    if _sol_loss is not None:
                        _sol_loss.backward()
                        total_loss_val += _sol_loss.item()
                        _q_total_loss_val += _sol_loss.item()
                        n_groups += 1
                    else:
                        skipped += 1
                        _skipped_zero_var += 1

                _q_ids_v   = [t[1] for t in _valid_q]
                _q_masks_v = [t[2] for t in _valid_q]
                _q_olps_v  = [t[3] for t in _valid_q]

                _q_loss = grpo_loss_for_group(
                    model=model,
                    input_ids_list=_q_ids_v,
                    response_masks=_q_masks_v,
                    rewards=_question_agg_rewards,
                    old_log_probs=_q_olps_v,
                    clip_eps=args.clip_eps,
                    kl_coef=0.0,
                    ref_model=None,
                )
                if _q_loss is not None:
                    _q_loss.backward()
                    logger.debug(
                        "Q-GRPO: loss=%.4f q_rewards=%s (variance=%.4f)",
                        _q_loss.item(),
                        [f"{r:.3f}" for r in _question_agg_rewards],
                        float(np.var(_question_agg_rewards)),
                    )

                if any(r > 0.5 for r in _question_agg_rewards):
                    q_quality_good += 1

                _mean_r_sp = float(np.mean(all_rewards[-len(_valid_q)*args.group_size:])) if all_rewards else 0.0
                _q_acc_pct = 100.0 * q_quality_good / max(1, n_self_play)
                pbar.set_postfix(
                    loss=f"{_q_total_loss_val / max(1, len(_valid_q)):.4f}",
                    mean_r=f"{_mean_r_sp:.3f}",
                    q_acc=f"{_q_acc_pct:.0f}%",
                    q_rew=f"{float(np.mean(all_q_rewards)):.3f}" if all_q_rewards else "n/a",
                    skip=skipped,
                )
                continue

            question = generate_question(
                model=model,
                tokenizer=tokenizer,
                instruction=instruction,
                max_new_tokens=128,
                device=device,
                temperature=min(0.90, _annealed_temp + 0.05),
            )
            if len(question.strip()) < 10:
                logger.debug(
                    "Self-play: generated question too short (%d chars), skipping group.",
                    len(question.strip()),
                )
                skipped += 1
                continue
            q_gen_valid += 1
            n_self_play += 1
            gold = None
        else:
            question = qa["question"]
            gold = qa["gold_final"]
            target_topic = "grounded"
            target_difficulty = 0.5

        solution_prompt = math_env.format_solution_prompt(question)
        solutions, input_ids_list, response_masks, old_log_probs_list = (
            generate_solutions_batched(
                model=model,
                tokenizer=tokenizer,
                prompt=solution_prompt,
                K=args.group_size,
                max_new_tokens=args.max_new_tokens,
                temperature=_annealed_temp,
                device=device,
            )
        )

        if args.overlong_filter:
            _valid = [
                (sol, ids, mask, olp)
                for sol, ids, mask, olp
                in zip(solutions, input_ids_list, response_masks, old_log_probs_list)
                if int(mask.sum().item()) < args.max_new_tokens
            ]
            if _valid:
                solutions, input_ids_list, response_masks, old_log_probs_list = (
                    zip(*_valid)  # type: ignore[assignment]
                )
                solutions        = list(solutions)
                input_ids_list   = list(input_ids_list)
                response_masks   = list(response_masks)
                old_log_probs_list = list(old_log_probs_list)
            else:
                skipped += 1
                continue

        rewards = []
        _sp_q_rew_this_group: List[float] = []
        for sol in solutions:
            if use_self_play:
                r, q_rew, _, q_met = compute_self_play_reward(
                    question=question,
                    solution=sol,
                    target_topic=target_topic,
                    target_difficulty=target_difficulty,
                    math_env=math_env,
                )
                _sp_q_rew_this_group.append(q_rew)
                all_q_rewards.append(q_rew)
                _qc_topic.append(q_met["topic_match"])
                _qc_diff.append(q_met["difficulty_fit"])
                _qc_clarity.append(q_met["clarity"])
                _qc_novelty.append(q_met["novelty"])
                _qc_solvability.append(q_met["solvability"])
                _sp_ci = q_met.get("sp_chain_integrity_score")
                if _sp_ci is not None:
                    _sp_chain_scores.append(float(_sp_ci))
            else:
                r_dict = compute_grounded_reward(
                    question=question,
                    solution=sol,
                    gold_final=gold,
                    math_env=math_env,
                )
                r = r_dict["combined_score"]
                _grounded_step_accs.append(r_dict["step_accuracy"])
                _grounded_lccps.append(r_dict["lccp"])
                _grounded_gt_matches.append(bool(r_dict["gt_match"]))
                if r_dict.get("chain_arith_score") is not None:
                    _chain_arith_scores.append(float(r_dict["chain_arith_score"]))
                if r_dict.get("chain_dep_score") is not None:
                    _chain_dep_scores.append(float(r_dict["chain_dep_score"]))
                if r_dict.get("chain_integrity_score") is not None:
                    _chain_integrity_scores.append(float(r_dict["chain_integrity_score"]))

                _shadow_extract_counter += 1
                if (
                    _phase == _Phase.SELFPLAY_RAMP
                    and not _use_chain_as_primary
                    and _unified_calc is not None
                    and _shadow_extract_counter % _SHADOW_EVERY == 0
                ):
                    _prm_ps = (
                        0.60 * r_dict.get("prm_final_score", 0.0)
                        + 0.40 * r_dict.get("prm_mean_score", 0.0)
                    )
                    try:
                        _shadow = _unified_calc.compute(
                            solution=sol,
                            gold_answer=gold,
                            question=question,
                            topic=target_topic,
                            phase="grounded",
                        )
                        _rolling_chain_scores.append(_shadow.chain_integrity_score)
                        _rolling_prm_scores.append(_prm_ps)
                        _rolling_successes.append(1 if _shadow.extraction_succeeded else 0)
                    except Exception:
                        _rolling_successes.append(0)
            rewards.append(r)
        all_rewards.extend(rewards)
        if use_self_play:
            _sp_rewards.extend(rewards)
        else:
            _grounded_rewards.extend(rewards)

        if use_self_play and _sp_q_rew_this_group:
            if float(np.mean(_sp_q_rew_this_group)) > 0.5:
                q_quality_good += 1

        if use_self_play:
            if not _verify_self_play_answer(solutions, target_topic, target_difficulty):
                skipped += 1
                continue

        if not use_self_play:
            _key = _question_key(question)
            _q_attempts[_key] += len(solutions)
            _group_median = float(np.median(rewards))
            _q_wins[_key] += sum(1 for r in rewards if r > _group_median)

        _reward_std = float(np.std(rewards))
        if _reward_std < 0.02:
            skipped += 1
            _skipped_zero_var += 1
            _pf_zv: Dict = dict(mean_r=f"{np.mean(rewards):.3f}", skip=skipped, loss="0var")
            pbar.set_postfix(**_pf_zv)
            continue

        group_loss = grpo_loss_for_group(
            model=model,
            input_ids_list=input_ids_list,
            response_masks=response_masks,
            rewards=rewards,
            old_log_probs=old_log_probs_list,
            clip_eps=args.clip_eps,
            kl_coef=args.kl_coef,
            ref_model=ref_model,
        )

        if group_loss is None:
            skipped += 1
            _skipped_zero_var += 1
            _pf: Dict = dict(mean_r=f"{np.mean(rewards):.3f}", skip=skipped, loss="skip")
            if n_self_play > 0 and all_q_rewards:
                _q_acc_pct = 100.0 * q_quality_good / max(1, n_self_play)
                _pf["q_acc"] = f"{_q_acc_pct:.0f}%"
            pbar.set_postfix(**_pf)
            continue

        group_loss.backward()
        total_loss_val += group_loss.item()
        n_groups += 1
        _pf = dict(
            mean_r=f"{np.mean(rewards):.3f}",
            loss=f"{group_loss.item():.4f}",
            skip=skipped,
        )
        if n_self_play > 0 and all_q_rewards:
            _q_acc_pct = 100.0 * q_quality_good / max(1, n_self_play)
            _pf["q_acc"] = f"{_q_acc_pct:.0f}%"
            _pf["q_rew"]  = f"{float(np.mean(all_q_rewards)):.3f}"
        pbar.set_postfix(**_pf)

    if n_groups > 0:
        if n_groups > 1:
            for p in model.parameters():
                if p.grad is not None:
                    p.grad.div_(n_groups)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            args.max_grad_norm,
        )
        optimizer.step()
        loss_val = total_loss_val / n_groups
    else:
        loss_val = 0.0
    scheduler.step()

    iter_time = time.perf_counter() - iter_start
    mean_r   = float(np.mean(all_rewards))             if all_rewards   else 0.0
    std_r    = float(np.std(all_rewards))              if all_rewards   else 0.0
    acc_r    = float(np.mean([r > 0.5 for r in all_rewards])) if all_rewards else 0.0
    grounded_acc_r = (
        float(np.mean([r > 0.5 for r in _grounded_rewards]))
        if _grounded_rewards else 0.0
    )
    mean_step_acc = (
        float(np.mean(_grounded_step_accs))
        if _grounded_step_accs else 0.0
    )
    mean_lccp = (
        float(np.mean(_grounded_lccps))
        if _grounded_lccps else 0.0
    )
    mean_q_r = float(np.mean(all_q_rewards)) if all_q_rewards else 0.0

    mean_chain_arith     = float(np.mean(_chain_arith_scores))     if _chain_arith_scores     else None
    mean_chain_dep       = float(np.mean(_chain_dep_scores))       if _chain_dep_scores       else None
    mean_chain_integrity = float(np.mean(_chain_integrity_scores)) if _chain_integrity_scores else None
    mean_sp_chain        = float(np.mean(_sp_chain_scores))        if _sp_chain_scores        else None

    gt_match_rate = (
        float(sum(_grounded_gt_matches) / len(_grounded_gt_matches))
        if _grounded_gt_matches else 0.0
    )

    if _phase == _Phase.GROUNDED_ONLY:
        _graduation_ready = (
            gt_match_rate    >= args.selfplay_gt_thresh
            and grounded_acc_r >= args.selfplay_grounded_thresh
            and mean_step_acc  >= args.selfplay_step_thresh
            and iteration      >= args.min_warmup
        )
        if _graduation_ready:
            _phase = _Phase.SELFPLAY_RAMP
            logger.info(
                "PHASE \u2192 SELFPLAY_RAMP at iter %d "
                "(gt_match=%.2f grounded_acc=%.2f step_acc=%.2f) \u2014 "
                "shadow extraction active; chain scoring deferred until "
                "calibration passes (corr\u22650.70, success_rate\u22650.80)",
                iteration, gt_match_rate, grounded_acc_r, mean_step_acc,
            )
    elif _phase in (_Phase.SELFPLAY_RAMP, _Phase.CONTINUOUS):
        _selfplay_iterations += 1
        if _phase == _Phase.SELFPLAY_RAMP and _selfplay_iterations >= args.selfplay_ramp_iters:
            _phase = _Phase.CONTINUOUS
            logger.info(
                "PHASE \u2192 CONTINUOUS at iter %d (ramp complete after %d iters)",
                iteration, _selfplay_iterations,
            )

        if len(_rolling_chain_scores) > _CALIB_MAX:
            _rolling_chain_scores = _rolling_chain_scores[-_CALIB_MAX:]
            _rolling_prm_scores   = _rolling_prm_scores[-_CALIB_MAX:]
            _rolling_successes    = _rolling_successes[-_CALIB_MAX:]

        if not _use_chain_as_primary and len(_rolling_chain_scores) >= _CALIB_WINDOW:
            from scipy.stats import pearsonr
            try:
                _r, _ = pearsonr(
                    _rolling_chain_scores[-_CALIB_WINDOW:],
                    _rolling_prm_scores[-_CALIB_WINDOW:],
                )
                _chain_prm_correlation = float(_r)
            except Exception:
                _chain_prm_correlation = 0.0
            _rolling_n = len(_rolling_successes[-_CALIB_WINDOW:])
            _extraction_success_rate = (
                sum(_rolling_successes[-_CALIB_WINDOW:]) / _rolling_n
                if _rolling_n > 0 else 0.0
            )
            if (
                _chain_prm_correlation >= 0.70
                and _extraction_success_rate >= 0.80
            ):
                _use_chain_as_primary = True
                math_env.use_chain_scoring = True
                logger.info(
                    "CHAIN PRIMARY activated at iter %d: "
                    "corr=%.2f extraction_rate=%.2f (window=%d) \u2014 "
                    "unified calculator now drives reward scoring",
                    iteration, _chain_prm_correlation,
                    _extraction_success_rate, _CALIB_WINDOW,
                )
            else:
                logger.debug(
                    "Chain calibration: corr=%.2f success_rate=%.2f "
                    "(need corr\u22650.70, success\u22650.80; window=%d/%d)",
                    _chain_prm_correlation, _extraction_success_rate,
                    len(_rolling_chain_scores), _CALIB_WINDOW,
                )

        _prev_suspended = _selfplay_suspended
        _selfplay_suspended = (
            bool(_grounded_gt_matches) and gt_match_rate < args.grounded_floor
        )
        if _selfplay_suspended and not _prev_suspended:
            logger.warning(
                "GROUNDED FLOOR: gt_match_rate=%.2f fell below floor=%.2f \u2014 "
                "suspending self-play for recovery",
                gt_match_rate, args.grounded_floor,
            )
        elif not _selfplay_suspended and _prev_suspended:
            logger.info(
                "GROUNDED FLOOR: gt_match_rate=%.2f recovered above floor=%.2f \u2014 "
                "resuming self-play",
                gt_match_rate, args.grounded_floor,
            )

    q_gen_valid_rate = (q_gen_valid   / q_gen_attempts)  if q_gen_attempts  > 0 else 0.0
    q_quality_rate   = (q_quality_good / n_self_play)    if n_self_play     > 0 else 0.0
    mean_q_topic     = float(np.mean(_qc_topic))       if _qc_topic      else 0.0
    mean_q_diff      = float(np.mean(_qc_diff))        if _qc_diff       else 0.0
    mean_q_clarity   = float(np.mean(_qc_clarity))     if _qc_clarity    else 0.0
    mean_q_novelty   = float(np.mean(_qc_novelty))     if _qc_novelty    else 0.0
    mean_q_solvab    = float(np.mean(_qc_solvability)) if _qc_solvability else 0.0

    _cur_lr = optimizer.param_groups[0]["lr"]

    logger.info(
        "Iter %d | loss=%.4f | reward mean=%.3f std=%.3f | "
        "gt_match=%.1f%% | grounded_acc=%.1f%% | step_acc=%.1f%% | lccp=%.1f%% | "
        "batch_acc=%.1f%% | phase=%s sp_ratio=%.0f%% | "
        "groups=%d skipped=%d(0var=%d) | lr=%.2e | %.1fs",
        iteration, loss_val, mean_r, std_r,
        100 * gt_match_rate,
        100 * grounded_acc_r,
        100 * mean_step_acc,
        100 * mean_lccp,
        100 * acc_r,
        _phase.name, 100 * _effective_sp_ratio,
        n_groups, skipped, _skipped_zero_var, _cur_lr, iter_time,
    )
    _total_attempted = n_groups + skipped
    if _total_attempted > 0 and _skipped_zero_var / _total_attempted > 0.30:
        logger.warning(
            "STARVATION: %.0f%% of groups skipped (zero variance). "
            "grounded_acc=%.1f%% suggests curriculum is %s. "
            "Consider adjusting --difficulty-alpha.",
            100 * _skipped_zero_var / _total_attempted,
            100 * grounded_acc_r,
            "too easy (raise alpha)" if grounded_acc_r > 0.75 else "too hard (lower alpha)",
        )

    if n_self_play > 0:
        logger.info(
            "  Question generation: %d/%d valid (%.0f%%) | "
            "q_reward=%.3f | q_acc=%.1f%% (>0.5 quality) | "
            "topic=%.2f diff=%.2f clarity=%.2f novelty=%.2f solvability=%.2f",
            q_gen_valid, q_gen_attempts, 100 * q_gen_valid_rate,
            mean_q_r, 100 * q_quality_rate,
            mean_q_topic, mean_q_diff, mean_q_clarity,
            mean_q_novelty, mean_q_solvab,
        )

    iter_metrics: Dict = {
        "iteration":             iteration,
        "loss":                  loss_val,
        "mean_reward":           mean_r,
        "std_reward":            std_r,
        "batch_accuracy":        acc_r,
        "grounded_accuracy":     grounded_acc_r,
        "gt_match_rate":         round(gt_match_rate, 4),
        "step_accuracy":         mean_step_acc,
        "lccp":                  mean_lccp,
        "n_groups":              n_groups,
        "skipped_groups":        skipped,
        "learning_rate":         _cur_lr,
        "iter_time_s":           iter_time,
        "training_phase":        _phase.name,
        "effective_sp_ratio":    round(_effective_sp_ratio, 3),
        "selfplay_suspended":    int(_selfplay_suspended),
        "chain_arith_score":       round(mean_chain_arith, 4)     if mean_chain_arith     is not None else None,
        "chain_dep_score":         round(mean_chain_dep, 4)       if mean_chain_dep       is not None else None,
        "chain_integrity_score":   round(mean_chain_integrity, 4) if mean_chain_integrity is not None else None,
        "sp_chain_integrity_score": round(mean_sp_chain, 4)       if mean_sp_chain        is not None else None,
        "chain_prm_correlation":   round(_chain_prm_correlation, 3),
        "extraction_success_rate": round(_extraction_success_rate, 3),
        "chain_scoring_active":    int(_use_chain_as_primary),
        "n_self_play_groups":    n_self_play,
        "q_gen_attempts":        q_gen_attempts,
        "q_gen_valid":           q_gen_valid,
        "q_gen_valid_rate":      round(q_gen_valid_rate, 4),
        "mean_question_reward":  round(mean_q_r, 4),
        "q_quality_rate":        round(q_quality_rate, 4),
        "q_topic_match":         round(mean_q_topic,   4),
        "q_difficulty_fit":      round(mean_q_diff,    4),
        "q_clarity":             round(mean_q_clarity, 4),
        "q_novelty":             round(mean_q_novelty, 4),
        "q_solvability":         round(mean_q_solvab,  4),
    }

    if iteration % args.eval_every == 0:
        _eval_ds_label = _infer_eval_dataset_name(args.eval_data_path)
        logger.info("Evaluating %s (%d samples)...", _eval_ds_label, args.eval_max_samples)
        eval_res = evaluate_policy(
            model, tokenizer,
            args.eval_data_path, args.eval_max_samples, args.eval_max_new_tokens,
            math_env=math_env,
            pass_at_k=args.eval_pass_at_k,
        )
        cur_combined = float(eval_res.get("combined_score", best_combined))
        cur_prm_mean = float(eval_res.get("prm_mean",       best_prm_mean))
        _log_eval_result(f"iter {iteration}", eval_res, best=best_combined)
        if cur_combined > best_combined + 1e-4:
            reason = f"combined {cur_combined:.4f} > {best_combined:.4f}"
            best_combined  = cur_combined
            best_prm_mean  = max(best_prm_mean, cur_prm_mean)
            best_accuracy  = best_combined
            best_path = out_dir / "best_policy"
            model.save_pretrained(str(best_path))
            tokenizer.save_pretrained(str(best_path))
            logger.info("New best saved \u2192 %s  (%s)", best_path, reason)
        iter_metrics.update(eval_res)

    is_last_iter = iteration == args.num_iterations
    should_save = is_last_iter or (
        args.save_every > 0 and iteration % args.save_every == 0
    )
    if should_save:
        ckpt_path = out_dir / f"iter_{iteration:04d}"
        ckpt_path.mkdir(exist_ok=True)
        model.save_pretrained(str(ckpt_path))
        tokenizer.save_pretrained(str(ckpt_path))
        if args.keep_last and args.keep_last > 0:
            existing = sorted(
                p for p in out_dir.iterdir()
                if p.is_dir() and p.name.startswith("iter_")
            )
            to_remove = existing[: -args.keep_last]
            for old in to_remove:
                try:
                    shutil.rmtree(old)
                    logger.info("Pruned old checkpoint: %s", old.name)
                except OSError as exc:
                    logger.warning("Could not prune %s: %s", old.name, exc)

    metrics_log.append(iter_metrics)
    (out_dir / "metrics.jsonl").write_text(
        "\n".join(json.dumps(m) for m in metrics_log), encoding="utf-8"
    )
    _append_metrics_csv({
        "iteration":      iter_metrics["iteration"],
        "timestamp":      datetime.now().isoformat(timespec="seconds"),
        "loss":           iter_metrics.get("loss", 0.0),
        "mean_reward":    iter_metrics.get("mean_reward", 0.0),
        "std_reward":     iter_metrics.get("std_reward", 0.0),
        "batch_accuracy": iter_metrics.get("batch_accuracy", 0.0),
        "n_groups":       iter_metrics.get("n_groups", 0),
        "skipped_groups": iter_metrics.get("skipped_groups", 0),
        "learning_rate":  iter_metrics.get("learning_rate", 0.0),
        "iter_time_s":    iter_metrics.get("iter_time_s", 0.0),
        "gsm8k_combined":   iter_metrics.get("combined_score",        ""),
        "gsm8k_correct_rt": iter_metrics.get("correct_rate",          ""),
        "gsm8k_prm":        iter_metrics.get("prm_mean",              ""),
        "gsm8k_sympy":      iter_metrics.get("sympy_mean",            ""),
        "gsm8k_format":     iter_metrics.get("format_mean",           ""),
        "gsm8k_n_scored":   iter_metrics.get("n_scored",              ""),
        "gsm8k_final_ans":  iter_metrics.get("final_answer_accuracy", ""),
    })

In [ ]:
logger.info("=" * 70)
logger.info("GRPO training complete.")
logger.info(
    "Best training-objective score : %.4f  "
    "(0.50\u00d7correct + 0.40\u00d7process[0.60\u00d7prm_final+0.40\u00d7prm_mean] + 0.10\u00d7fmt)",
    best_combined,
)
logger.info("Best PRM component mean       : %.3f", best_prm_mean)
logger.info("Checkpoints                   : %s", out_dir)
logger.info("Logs                          : %s", log_dir)
logger.info("Console log                   : %s", console_log_path)
logger.info("=" * 70)

summary: Dict[str, Any] = {
    "run_name":          run_name,
    "best_accuracy":     best_combined,
    "best_combined":     best_combined,
    "best_prm_mean":     best_prm_mean,
    "total_iterations":  args.num_iterations,
    "checkpoints_dir":   str(out_dir),
    "log_dir":           str(log_dir),
    "console_log":       str(console_log_path),
    "metrics_csv":       str(_metrics_csv_path),
    "metrics_jsonl":     str(out_dir / "metrics.jsonl"),
}
(log_dir / "summary.json").write_text(
    json.dumps(summary, indent=2, default=str), encoding="utf-8"
)
logger.info("Summary written to %s", log_dir / "summary.json")

_metrics_jsonl = out_dir / "metrics.jsonl"
if _metrics_jsonl.exists():
    try:
        import importlib
        if importlib.util.find_spec("matplotlib") is None:
            logger.warning(
                "matplotlib not installed \u2014 skipping auto-plot. "
                "Install with: pip install matplotlib  then run: "
                "python scripts/plot_grpo_run.py %s",
                _metrics_jsonl,
            )
        else:
            from scripts.plot_grpo_run import generate_plots as _gen_plots
            _plot_dir = _gen_plots(_metrics_jsonl)
            logger.info("Plots saved \u2192 %s", _plot_dir)
    except Exception as _plot_exc:
        logger.warning(
            "Plot generation failed (%s: %s). "
            "Run manually: python scripts/plot_grpo_run.py %s",
            type(_plot_exc).__name__, _plot_exc, _metrics_jsonl,
        )

_teardown_logging()